## BUSINESS UNDERSTANDING

### BACKGROUND

The FIFA World Cup is the world's most-watched sporting event, with over 3.5 billion viewers globally. The 2026 tournament is particularly significant as it marks:

The first 48-team tournament (expanded from 32 teams)

Hosted across 3 countries: USA, Canada, and Mexico

104 matches (up from 64) across 16 host cities

### BUSINESS CONTEXT

This challenge addresses the growing demand for data-driven insights in sports analytics. Stakeholders across the football ecosystem can benefit from predictive models that forecast tournament outcomes.

### DATA CONSTRAINTS
- Closed-data challenge: Only the Fjelstul World Cup Database
- No external data: No 2026 tournament information

- No web scraping: No betting odds, rankings, or real-time data


### OBJECTIVES
- Predict total goals for each of the 48 teams (RMSE, 60% weight)
- Predict tournament stage reached for each team (F1 Score, 40% weight)
- Minimize RMSE on the private leaderboard for goal predictions
- Classify each team into one of 7 stages: Group, Round32, Round16, QF, SF, Runner-up, Champion
- Maximize F1 Score on the private leaderboard for stage classification


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Load all files
data_dir = Path('data/')
all_files = list(data_dir.glob('*.csv'))

# Create dictionary of dataframes
dfs = {}
for file in all_files:
    name = file.stem
    dfs[name] = pd.read_csv(file)
    print(f"{name:25s} : {len(dfs[name]):5d} rows, {len(dfs[name].columns):2d} columns")

awards                    :     8 rows,  5 columns
award_winners             :   200 rows, 12 columns
bookings                  :  3178 rows, 26 columns
confederations            :     6 rows,  5 columns
goals                     :  3637 rows, 27 columns
groups                    :   159 rows,  7 columns
group_standings           :   626 rows, 19 columns
host_countries            :    31 rows,  7 columns
managers                  :   475 rows,  7 columns
manager_appearances       :  2538 rows, 17 columns
manager_appointments      :   637 rows, 10 columns
matches                   :  1248 rows, 37 columns
penalty_kicks             :   396 rows, 19 columns
players                   : 10401 rows, 13 columns
player_appearances        : 27432 rows, 21 columns
qualified_teams           :   625 rows,  8 columns
referees                  :   493 rows, 10 columns
referee_appearances       :  1248 rows, 15 columns
referee_appointments      :   668 rows, 10 columns
squads                    : 138

### Data Quality Assessment

In [2]:
def comprehensive_data_report(dfs):
    """Generate comprehensive data quality report"""
    report = []
    
    for name, df in dfs.items():
        # Basic info
        missing_pct = (df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100
        duplicate_rows = df.duplicated().sum()
        
        report.append({
            'Dataset': name,
            'Rows': df.shape[0],
            'Columns': df.shape[1],
            'Missing %': f"{missing_pct:.1f}%",
            'Duplicates': duplicate_rows,
            'Memory (MB)': df.memory_usage(deep=True).sum() / 1024**2
        })
    
    report_df = pd.DataFrame(report).sort_values('Missing %', ascending=False)
    return report_df

quality_report = comprehensive_data_report(dfs)
print("DATA QUALITY REPORT:")
display(quality_report)

# Identify datasets with high missing values
high_missing = quality_report[quality_report['Missing %'].str.rstrip('%').astype(float) > 5]
if len(high_missing) > 0:
    print("\n⚠️ Datasets with >5% missing values:")
    display(high_missing)

DATA QUALITY REPORT:


,Dataset,Rows,Columns,Missing %,Duplicates,Memory (MB)
0,awards,8,5,0.0%,0,0.000675
1,award_winners,200,12,0.0%,0,0.034243
2,bookings,3178,26,0.0%,0,1.078738
3,confederations,6,5,0.0%,0,0.000914
4,goals,3637,27,0.0%,0,1.313055
5,groups,159,7,0.0%,0,0.016397
6,group_standings,626,19,0.0%,0,0.130340
7,host_countries,31,7,0.0%,0,0.003512
8,managers,475,7,0.0%,0,0.058593
9,manager_appearances,2538,17,0.0%,0,0.649355


In [ ]:
# Data directory
data_dir = Path('data/')

# Load all datasets (already done in your notebook)
# But let's organize them for easy access
dfs = {
    'awards': pd.read_csv(data_dir / 'awards.csv'),
    'award_winners': pd.read_csv(data_dir / 'award_winners.csv'),
    'bookings': pd.read_csv(data_dir / 'bookings.csv'),
    'confederations': pd.read_csv(data_dir / 'confederations.csv'),
    'goals': pd.read_csv(data_dir / 'goals.csv'),
    'groups': pd.read_csv(data_dir / 'groups.csv'),
    'group_standings': pd.read_csv(data_dir / 'group_standings.csv'),
    'host_countries': pd.read_csv(data_dir / 'host_countries.csv'),
    'managers': pd.read_csv(data_dir / 'managers.csv'),
    'manager_appearances': pd.read_csv(data_dir / 'manager_appearances.csv'),
    'manager_appointments': pd.read_csv(data_dir / 'manager_appointments.csv'),
    'matches': pd.read_csv(data_dir / 'matches.csv'),
    'penalty_kicks': pd.read_csv(data_dir / 'penalty_kicks.csv'),
    'players': pd.read_csv(data_dir / 'players.csv'),
    'player_appearances': pd.read_csv(data_dir / 'player_appearances.csv'),
    'qualified_teams': pd.read_csv(data_dir / 'qualified_teams.csv'),
    'referees': pd.read_csv(data_dir / 'referees.csv'),
    'referee_appearances': pd.read_csv(data_dir / 'referee_appearances.csv'),
    'referee_appointments': pd.read_csv(data_dir / 'referee_appointments.csv'),
    'squads': pd.read_csv(data_dir / 'squads.csv'),
    'stadiums': pd.read_csv(data_dir / 'stadiums.csv'),
    'substitutions': pd.read_csv(data_dir / 'substitutions.csv'),
    'teams': pd.read_csv(data_dir / 'teams.csv'),
    'team_appearances': pd.read_csv(data_dir / 'team_appearances.csv'),
    'tournaments': pd.read_csv(data_dir / 'tournaments.csv'),
    'tournament_stages': pd.read_csv(data_dir / 'tournament_stages.csv'),
    'tournament_standings': pd.read_csv(data_dir / 'tournament_standings.csv')
}

print("✅ All datasets loaded successfully!")
print(f"Total datasets: {len(dfs)}")

# Show dataset sizes
print("\n📊 Dataset Overview:")
overview = []
for name, df in dfs.items():
    overview.append({
        'Dataset': name,
        'Rows': df.shape[0],
        'Columns': df.shape[1],
        'Memory (MB)': df.memory_usage(deep=True).sum() / 1024**2
    })
overview_df = pd.DataFrame(overview).sort_values('Rows', ascending=False)
display(overview_df)

# Identify core datasets for prediction
core_datasets = ['team_appearances', 'teams', 'tournaments', 'matches', 'goals', 
                 'tournament_stages', 'tournament_standings', 'confederations', 'host_countries']
print("\n🎯 Core datasets for prediction:")
for ds in core_datasets:
    print(f"  • {ds}: {dfs[ds].shape[0]} rows, {dfs[ds].shape[1]} columns")

✅ All datasets loaded successfully!
Total datasets: 27

📊 Dataset Overview:


,Dataset,Rows,Columns,Memory (MB)
14,player_appearances,27432,21,8.041348
19,squads,13843,12,2.303242
13,players,10401,13,1.860174
21,substitutions,10222,24,3.305119
4,goals,3637,27,1.313055
2,bookings,3178,26,1.078738
9,manager_appearances,2538,17,0.649355
23,team_appearances,2496,36,1.083694
11,matches,1248,37,0.570442
17,referee_appearances,1248,15,0.343364



🎯 Core datasets for prediction:
  • team_appearances: 2496 rows, 36 columns
  • teams: 88 rows, 14 columns
  • tournaments: 30 rows, 18 columns
  • matches: 1248 rows, 37 columns
  • goals: 3637 rows, 27 columns
  • tournament_stages: 155 rows, 16 columns
  • tournament_standings: 120 rows, 7 columns
  • confederations: 6 rows, 5 columns
  • host_countries: 31 rows, 7 columns


### Inspect Core Datasets Necessary for my Analysis

In [32]:
# ============================================================
#  Inspect Core Datasets
# ============================================================

#  Inspect the datasets
core = ['tournament_standings', 'team_appearances', 'matches', 'goals', 
        'group_standings', 'qualified_teams', 'tournaments']

for name in core:
    print(f"\n{'='*50}")
    print(f"{name.upper()}")
    print(f"{'='*50}")
    print(f"Shape: {dfs[name].shape}")
    display(dfs[name].head(3))
    print(f"Columns: {list(dfs[name].columns)}")


TOURNAMENT_STANDINGS
Shape: (120, 7)


,key_id,tournament_id,tournament_name,position,team_id,team_name,team_code
0,1,WC-1930,1930 FIFA Men's World Cup,1,T-84,Uruguay,URY
1,2,WC-1930,1930 FIFA Men's World Cup,2,T-03,Argentina,ARG
2,3,WC-1930,1930 FIFA Men's World Cup,3,T-83,United States,USA


Columns: ['key_id', 'tournament_id', 'tournament_name', 'position', 'team_id', 'team_name', 'team_code']

TEAM_APPEARANCES
Shape: (2496, 36)


,key_id,tournament_id,tournament_name,match_id,match_name,stage_name,group_name,group_stage,knockout_stage,replayed,...,goals_against,goal_differential,extra_time,penalty_shootout,penalties_for,penalties_against,result,win,lose,draw
0,1,WC-1930,1930 FIFA Men's World Cup,M-1930-01,France vs Mexico,group stage,Group 1,1,0,0,...,1,3,0,0,0,0,win,1,0,0
1,2,WC-1930,1930 FIFA Men's World Cup,M-1930-01,France vs Mexico,group stage,Group 1,1,0,0,...,4,-3,0,0,0,0,lose,0,1,0
2,3,WC-1930,1930 FIFA Men's World Cup,M-1930-02,United States vs Belgium,group stage,Group 4,1,0,0,...,0,3,0,0,0,0,win,1,0,0


Columns: ['key_id', 'tournament_id', 'tournament_name', 'match_id', 'match_name', 'stage_name', 'group_name', 'group_stage', 'knockout_stage', 'replayed', 'replay', 'match_date', 'match_time', 'stadium_id', 'stadium_name', 'city_name', 'country_name', 'team_id', 'team_name', 'team_code', 'opponent_id', 'opponent_name', 'opponent_code', 'home_team', 'away_team', 'goals_for', 'goals_against', 'goal_differential', 'extra_time', 'penalty_shootout', 'penalties_for', 'penalties_against', 'result', 'win', 'lose', 'draw']

MATCHES
Shape: (1248, 37)


...

Columns: ['key_id', 'tournament_id', 'tournament_name', 'stage_number', 'stage_name', 'group_name', 'position', 'team_id', 'team_name', 'team_code', 'played', 'wins', 'draws', 'losses', 'goals_for', 'goals_against', 'goal_difference', 'points', 'advanced']

QUALIFIED_TEAMS
Shape: (625, 8)


,key_id,tournament_id,tournament_name,team_id,team_name,team_code,count_matches,performance
0,1,WC-1930,1930 FIFA Men's World Cup,T-03,Argentina,ARG,5,final
1,2,WC-1930,1930 FIFA Men's World Cup,T-06,Belgium,BEL,2,group stage
2,3,WC-1930,1930 FIFA Men's World Cup,T-07,Bolivia,BOL,2,group stage


Columns: ['key_id', 'tournament_id', 'tournament_name', 'team_id', 'team_name', 'team_code', 'count_matches', 'performance']

TOURNAMENTS
Shape: (30, 18)


,key_id,tournament_id,tournament_name,year,start_date,end_date,host_country,winner,host_won,count_teams,group_stage,second_group_stage,final_round,round_of_16,quarter_finals,semi_finals,third_place_match,final
0,1,WC-1930,1930 FIFA Men's World Cup,1930,1930-07-13,1930-07-30,Uruguay,Uruguay,1,13,1,0,0,0,0,1,0,1
1,2,WC-1934,1934 FIFA Men's World Cup,1934,1934-05-27,1934-06-10,Italy,Italy,1,16,0,0,0,1,1,1,1,1
2,3,WC-1938,1938 FIFA Men's World Cup,1938,1938-06-04,1938-06-19,France,Italy,0,15,0,0,0,1,1,1,1,1


Columns: ['key_id', 'tournament_id', 'tournament_name', 'year', 'start_date', 'end_date', 'host_country', 'winner', 'host_won', 'count_teams', 'group_stage', 'second_group_stage', 'final_round', 'round_of_16', 'quarter_finals', 'semi_finals', 'third_place_match', 'final']


### Understand the Target Variable

In [33]:
# ============================================================
# Understand Your Target Variables
# ============================================================

qt = dfs['qualified_teams'].copy()

print("'performance' column (Stage Target) - unique values:")
print(qt['performance'].value_counts())

print(f"\n Total team-tournament records: {len(qt)}")
print(f"Tournaments covered: {qt['tournament_id'].nunique()}")
print(f"Unique teams: {qt['team_id'].nunique()}")

'performance' column (Stage Target) - unique values:
performance
group stage           278
round of 16           111
quarter-finals         68
final                  58
third-place match      56
quarter-final          32
second group stage     16
final round             4
semi-finals             2
Name: count, dtype: int64

 Total team-tournament records: 625
Tournaments covered: 30
Unique teams: 88


###  Fix Champion vs Runner-up Using tournament_standings

In [35]:
# ============================================================
# Fix Champion vs Runner-up
# ============================================================

standings = dfs['tournament_standings'].copy()

# Position 1 = Champion, Position 2 = Runner-up
winners = standings[standings['position'] == 1][['tournament_id', 'team_id']].copy()
winners['stage_target'] = 'champion'

runners_up = standings[standings['position'] == 2][['tournament_id', 'team_id']].copy()
runners_up['stage_target'] = 'runnerup'

# Override in qt
qt = qt.merge(
    standings[['tournament_id', 'team_id', 'position']],
    on=['tournament_id', 'team_id'],
    how='left'
)

qt.loc[qt['position'] == 1, 'stage_target'] = 'champion'
qt.loc[qt['position'] == 2, 'stage_target'] = 'runnerup'

print("✅ Final stage distribution:")
print(qt['stage_target'].value_counts())

✅ Final stage distribution:
stage_target
group        278
roundof16    111
qf            68
runnerup      30
champion      30
sf             2
Name: count, dtype: int64


### Build Goals Features from team_appearances

In [ ]:
# ============================================================
# CELL 6: Build Features from team_appearances
# ============================================================

ta = dfs['team_appearances'].copy()

team_stats = (
    ta.groupby(['tournament_id', 'team_id'])
    .agg(
        matches_played        = ('match_id', 'count'),
        total_goals_scored    = ('goals_for', 'sum'),
        total_goals_conceded  = ('goals_against', 'sum'),
        total_wins            = ('win', 'sum'),
        total_losses          = ('lose', 'sum'),
        total_draws           = ('draw', 'sum'),
        went_to_extra_time    = ('extra_time', 'sum'),
        penalty_shootouts     = ('penalty_shootout', 'sum'),
    )
    .reset_index()
)

team_stats['goal_difference']    = team_stats['total_goals_scored'] - team_stats['total_goals_conceded']
team_stats['goals_per_match']    = team_stats['total_goals_scored'] / team_stats['matches_played']
team_stats['conceded_per_match'] = team_stats['total_goals_conceded'] / team_stats['matches_played']
team_stats['win_rate']           = team_stats['total_wins'] / team_stats['matches_played']

print(f"✅ Team stats shape: {team_stats.shape}")
display(team_stats.sort_values('total_goals_scored', ascending=False).head(10))

✅ Team stats shape: (625, 14)


,tournament_id,team_id,matches_played,total_goals_scored,total_goals_conceded,total_wins,total_losses,total_draws,went_to_extra_time,penalty_shootouts,goal_difference,goals_per_match,conceded_per_match,win_rate
63,WC-1954,T-36,5,27,10,4,1,0,1,0,17,5.400000,2.000000,0.800000
592,WC-2019,T-83,7,26,3,7,0,0,0,0,23,3.714286,0.428571,1.000000
252,WC-1991,T-83,6,25,5,6,0,0,0,0,20,4.166667,0.833333,1.000000
375,WC-2003,T-31,6,25,4,6,0,0,1,0,21,4.166667,0.666667,1.000000
71,WC-1954,T-86,6,25,14,5,1,0,0,0,11,4.166667,2.333333,0.833333
78,WC-1958,T-30,6,23,15,4,2,0,0,0,8,3.833333,2.500000,0.666667
286,WC-1995,T-53,6,23,1,6,0,0,0,0,22,3.833333,0.166667,1.000000
45,WC-1950,T-09,6,22,6,4,1,1,0,0,16,3.666667,1.000000,0.666667
424,WC-2007,T-31,6,21,0,5,0,1,0,0,21,3.500000,0.000000,0.833333
523,WC-2015,T-31,7,20,6,4,2,1,2,1,14,2.857143,0.857143,0.571429


In [ ]:
# ============================================================
# DEFINITIVE STAGE MAPPING - Covers all 625 rows
# ============================================================

qt = dfs['qualified_teams'].copy()

# Exact mapping based on actual values seen
stage_mapping = {
    'group stage':        'group',
    'second group stage': 'group',       
    'final round':        'group',      
    'round of 16':        'roundof16',
    'quarter-final':      'qf',         
    'quarter-finals':     'qf',          
    'third-place match':  'sf',          
    'final':              'runnerup',    
}

qt['stage_target'] = qt['performance'].str.strip().map(stage_mapping)

# ── Fix champions using tournament_standings ────────────────
standings = dfs['tournament_standings'].copy()
champions = standings[standings['position'] == 1][['tournament_id', 'team_id']]

qt = qt.merge(champions, on=['tournament_id', 'team_id'], how='left', indicator=True)
qt.loc[qt['_merge'] == 'both', 'stage_target'] = 'champion'
qt.drop(columns=['_merge'], inplace=True)

# ── Verify ──────────────────────────────────────────────────
print("✅ Stage distribution:")
print(qt['stage_target'].value_counts())

total = len(qt)
nulls = qt['stage_target'].isna().sum()
print(f"\n✅ Total rows : {total}")
print(f"✅ Nulls remaining: {nulls}")
print(f"✅ Coverage: {((total - nulls)/total)*100:.1f}%")

✅ Stage distribution:
stage_target
group        297
roundof16    111
qf           100
sf            58
champion      30
runnerup      29
Name: count, dtype: int64

✅ Total rows : 625
✅ Nulls remaining: 0
✅ Coverage: 100.0%


In [52]:
# ============================================================
# MASTER TABLE BUILD - Full Merge
# ============================================================

ta = dfs['team_appearances'].copy()
tournaments = dfs['tournaments'].copy()
group_standings = dfs['group_standings'].copy()
confederations = dfs['confederations'].copy()
teams = dfs['teams'].copy()

# ── Step 1: Goals & match features per team per tournament ──
team_stats = (
    ta.groupby(['tournament_id', 'team_id'])
    .agg(
        matches_played        = ('match_id', 'count'),
        total_goals_scored    = ('goals_for', 'sum'),
        total_goals_conceded  = ('goals_against', 'sum'),
        total_wins            = ('win', 'sum'),
        total_losses          = ('lose', 'sum'),
        total_draws           = ('draw', 'sum'),
        extra_time_games      = ('extra_time', 'sum'),
        penalty_shootouts     = ('penalty_shootout', 'sum'),
    )
    .reset_index()
)

team_stats['goal_difference']    = team_stats['total_goals_scored'] - team_stats['total_goals_conceded']
team_stats['goals_per_match']    = team_stats['total_goals_scored'] / team_stats['matches_played']
team_stats['conceded_per_match'] = team_stats['total_goals_conceded'] / team_stats['matches_played']
team_stats['win_rate']           = team_stats['total_wins'] / team_stats['matches_played']

print(f"✅ team_stats shape: {team_stats.shape}")

# ── Step 2: Group stage features ────────────────────────────
group_feats = (
    group_standings
    .groupby(['tournament_id', 'team_id'])
    .agg(
        group_goals_for      = ('goals_for', 'sum'),
        group_goals_against  = ('goals_against', 'sum'),
        group_points         = ('points', 'sum'),
        group_wins           = ('wins', 'sum'),
        group_draws          = ('draws', 'sum'),
        group_losses         = ('losses', 'sum'),
        group_position       = ('position', 'min'),
        advanced_from_group  = ('advanced', 'max'),
    )
    .reset_index()
)

print(f"✅ group_feats shape: {group_feats.shape}")

# ── Step 3: Start master from qt ────────────────────────────
master = qt[['tournament_id', 'team_id', 'team_name', 
             'team_code', 'stage_target']].copy()

# ── Step 4: Merge team stats ─────────────────────────────── 
master = master.merge(team_stats, on=['tournament_id', 'team_id'], how='left')

# ── Step 5: Merge group features ────────────────────────────
master = master.merge(group_feats, on=['tournament_id', 'team_id'], how='left')

# ── Step 6: Merge tournament info ───────────────────────────
master = master.merge(
    tournaments[['tournament_id', 'year', 'host_country', 'count_teams',
                 'group_stage', 'round_of_16', 'quarter_finals', 
                 'semi_finals', 'final']],
    on='tournament_id', how='left'
)

# ── Step 7: Merge confederation via teams ───────────────────
master = master.merge(
    teams[['team_id', 'confederation_id']],
    on='team_id', how='left'
)

# ── Step 8: Host flag ────────────────────────────────────────
master['is_host'] = (
    master['team_name'].str.lower() == master['host_country'].str.lower()
).astype(int)

# ── Step 9: Verify ──────────────────────────────────────────
print(f"\n✅ Master table shape: {master.shape}")
print(f"\n🔍 Null check:")
nulls = master.isnull().sum()
print(nulls[nulls > 0] if nulls.any() else "  No nulls ✅")

print(f"\n🎯 Targets:")
print(f"  total_goals_scored — mean: {master['total_goals_scored'].mean():.2f}, max: {master['total_goals_scored'].max()}")
print(f"  stage_target:\n{master['stage_target'].value_counts()}")

display(master.head(10))

✅ team_stats shape: (625, 14)
✅ group_feats shape: (594, 10)

✅ Master table shape: (625, 35)

🔍 Null check:
group_goals_for        31
group_goals_against    31
group_points           31
group_wins             31
group_draws            31
group_losses           31
group_position         31
advanced_from_group    31
dtype: int64

🎯 Targets:
  total_goals_scored — mean: 5.82, max: 27
  stage_target:
stage_target
group        297
roundof16    111
qf           100
sf            58
champion      30
runnerup      29
Name: count, dtype: int64


,tournament_id,team_id,team_name,team_code,stage_target,matches_played,total_goals_scored,total_goals_conceded,total_wins,total_losses,...,year,host_country,count_teams,group_stage,round_of_16,quarter_finals,semi_finals,final,confederation_id,is_host
0,WC-1930,T-03,Argentina,ARG,runnerup,5,18,9,4,1,...,1930,Uruguay,13,1,0,0,1,1,CF-4,0
1,WC-1930,T-06,Belgium,BEL,group,2,0,4,0,2,...,1930,Uruguay,13,1,0,0,1,1,CF-6,0
2,WC-1930,T-07,Bolivia,BOL,group,2,0,8,0,2,...,1930,Uruguay,13,1,0,0,1,1,CF-4,0
3,WC-1930,T-09,Brazil,BRA,group,2,5,2,1,1,...,1930,Uruguay,13,1,0,0,1,1,CF-4,0
4,WC-1930,T-13,Chile,CHL,group,3,5,3,2,1,...,1930,Uruguay,13,1,0,0,1,1,CF-4,0
5,WC-1930,T-30,France,FRA,group,3,4,3,1,2,...,1930,Uruguay,13,1,0,0,1,1,CF-6,0
6,WC-1930,T-46,Mexico,MEX,group,3,4,13,0,3,...,1930,Uruguay,13,1,0,0,1,1,CF-3,0
7,WC-1930,T-55,Paraguay,PRY,group,2,1,3,1,1,...,1930,Uruguay,13,1,0,0,1,1,CF-4,0
8,WC-1930,T-56,Peru,PER,group,2,1,4,0,2,...,1930,Uruguay,13,1,0,0,1,1,CF-4,0
9,WC-1930,T-61,Romania,ROU,group,2,3,5,1,1,...,1930,Uruguay,13,1,0,0,1,1,CF-6,0


In [53]:
# ============================================================
# FIX: Handle 31 null group features (old format tournaments)
# ============================================================

# Flag old-format tournaments BEFORE filling nulls
master['no_group_stage'] = master['group_goals_for'].isna().astype(int)

print("🏛️  Old format tournament teams (no group stage):")
print(master[master['no_group_stage'] == 1][['year', 'team_name', 'stage_target']].head(10))
print(f"\nTotal: {master['no_group_stage'].sum()} teams across old tournaments")

# Fill group feature nulls with 0
group_cols = [
    'group_goals_for', 'group_goals_against', 'group_points',
    'group_wins', 'group_draws', 'group_losses',
    'group_position', 'advanced_from_group'
]
master[group_cols] = master[group_cols].fillna(0)

# ── Final verification ───────────────────────────────────────
print(f"\n✅ Master table shape: {master.shape}")
nulls = master.isnull().sum()
print(f"\n🔍 Null check:")
print(nulls[nulls > 0] if nulls.any() else "  No nulls ✅")

print(f"\n📊 Feature columns ({master.shape[1]}):")
for col in master.columns:
    print(f"  {col}")

🏛️  Old format tournament teams (no group stage):
    year       team_name stage_target
13  1934       Argentina    roundof16
14  1934         Austria           sf
15  1934         Belgium    roundof16
16  1934          Brazil    roundof16
17  1934  Czechoslovakia     runnerup
18  1934           Egypt    roundof16
19  1934          France    roundof16
20  1934         Germany           sf
21  1934         Hungary           qf
22  1934           Italy     champion

Total: 31 teams across old tournaments

✅ Master table shape: (625, 36)

🔍 Null check:
  No nulls ✅

📊 Feature columns (36):
  tournament_id
  team_id
  team_name
  team_code
  stage_target
  matches_played
  total_goals_scored
  total_goals_conceded
  total_wins
  total_losses
  total_draws
  extra_time_games
  penalty_shootouts
  goal_difference
  goals_per_match
  conceded_per_match
  win_rate
  group_goals_for
  group_goals_against
  group_points
  group_wins
  group_draws
  group_losses
  group_position
  advanced_from_g

In [58]:
master.info()

<class 'pandas.DataFrame'>
RangeIndex: 625 entries, 0 to 624
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   tournament_id         625 non-null    str    
 1   team_id               625 non-null    str    
 2   team_name             625 non-null    str    
 3   team_code             625 non-null    str    
 4   stage_target          625 non-null    str    
 5   matches_played        625 non-null    int64  
 6   total_goals_scored    625 non-null    int64  
 7   total_goals_conceded  625 non-null    int64  
 8   total_wins            625 non-null    int64  
 9   total_losses          625 non-null    int64  
 10  total_draws           625 non-null    int64  
 11  extra_time_games      625 non-null    int64  
 12  penalty_shootouts     625 non-null    int64  
 13  goal_difference       625 non-null    int64  
 14  goals_per_match       625 non-null    float64
 15  conceded_per_match    625 non-null

In [59]:
master.to_csv('combined_data.csv', index=False)

### Goals Distribution Analysis 